# Build a point-in-time fundamentals panel

## Goal

For each entity, period, metric and unit, retain the latest disclosed version available at the cutoff, with provenance and availability time.

This notebook uses synthetic teaching data, not a paper replication or production observations.

## Setup

Use a Python 3.10+ kernel and run all cells in order. Computation uses only the standard library, without keys, networking or extra data files. Open in an existing Jupyter environment.

Embedded inputs match inputs.json in the same download directory. Edit args in the next cell to experiment; preserve explicit times and units.

In [ ]:
import json

# Synthetic inputs; no credentials or network access.
bundle = json.loads("{\"version\":1,\"tutorial\":\"pit-fundamentals-panel\",\"identity\":\"synthetic\",\"args\":[[{\"entity\":\"DEMO\",\"period\":\"2024-12-31\",\"metric\":\"revenue\",\"unit\":\"CNY_million\",\"value\":100,\"version\":\"v1\",\"publishedAt\":\"2025-03-20T18:00:00+08:00\",\"firstSeenAt\":\"2025-03-20T18:05:00+08:00\"},{\"entity\":\"DEMO\",\"period\":\"2024-12-31\",\"metric\":\"revenue\",\"unit\":\"CNY_million\",\"value\":105,\"version\":\"v2\",\"publishedAt\":\"2025-04-10T18:00:00+08:00\",\"firstSeenAt\":\"2025-04-10T18:02:00+08:00\"}],\"2025-03-31T23:59:59+08:00\"],\"expected\":[{\"entity\":\"DEMO\",\"period\":\"2024-12-31\",\"metric\":\"revenue\",\"unit\":\"CNY_million\",\"value\":100,\"version\":\"v1\",\"publishedAt\":\"2025-03-20T18:00:00+08:00\",\"firstSeenAt\":\"2025-03-20T18:05:00+08:00\",\"availableAt\":\"2025-03-20T10:05:00.000Z\"}]}")
args = bundle["args"]
expected = bundle["expected"]
print(json.dumps(args, ensure_ascii=False, indent=2))

## Steps

### 1. Distinguish four clocks

Period-end describes the economic period; publication time describes release; first-seen time describes your acquisition; the cutoff bounds the information set. The example uses max(publishedAt, firstSeenAt) for availability in your data chain, avoiding fictional historical possession of later-backfilled records.

### 2. Preserve versions rather than overwrite

Retain entity, period, metric, unit and version as record identity. The synthetic report first says 100 and later revises to 105; keep both. Quarantine conflicting versions. Real inputs additionally require consolidation scope, cumulative versus standalone periods, accounting standards and currency.

### 3. Filter availability before selection

Exclude records available after the cutoff before choosing the latest publication in each group. The example refuses to guess between different versions with ambiguous publication order. Selecting the database's latest version first and filtering by period-end afterward leaks future information.

### 4. Check unreleased and revised cases

A cutoff before first disclosure returns nothing; between disclosures it returns 100; after the revision becomes available it returns 105. Date-only sources must not silently become midnight timestamps. Use an explicitly conservative next-session convention or require review.

### Method and assumptions

- Code cannot manufacture point-in-time evidence when historical versions or release times are missing.
- The function chooses versions within each period; it does not select the latest period or compute TTM.
- Missing records are not zero; actual availability and licensing require separate confirmation.

In [ ]:
def select_as_of(rows, cutoff):
    """Select the latest published eligible version, without overwriting inputs."""
    from datetime import datetime, timezone
    import math
    import re

    def parse(value):
        if not isinstance(value, str) or not re.search(r"T.*(Z|[+-]\d{2}:\d{2})$", value):
            raise ValueError("timezone_required")
        try:
            return datetime.fromisoformat(value.replace("Z", "+00:00"))
        except ValueError:
            raise ValueError("timezone_required") from None

    boundary = parse(cutoff)
    eligible, identities, publication_orders = {}, set(), set()
    for row in rows:
        value = row.get("value")
        if (not all(row.get(field) for field in ("entity", "period", "metric", "unit"))
                or type(value) not in (int, float) or not math.isfinite(value)):
            raise ValueError("invalid_record")
        published, observed = parse(row.get("publishedAt")), parse(row.get("firstSeenAt"))
        key = tuple(row[field] for field in ("entity", "period", "metric", "unit"))
        identity = (key, row.get("version"))
        if not row.get("version") or identity in identities:
            raise ValueError("duplicate_or_missing_version")
        identities.add(identity)
        available = max(published, observed)
        if available > boundary:
            continue
        publication_order = (key, published)
        if publication_order in publication_orders:
            raise ValueError("ambiguous_publication_order")
        publication_orders.add(publication_order)
        if key not in eligible or published > eligible[key][0]:
            timestamp = available.astimezone(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")
            eligible[key] = (published, dict(row, availableAt=timestamp))
    return [eligible[key][1] for key in sorted(eligible)]


### Run the sample

At the synthetic 2025-03-31 cutoff, return v1 and 100, excluding April's v2. This verifies the example rule, not completeness of vendor revision history.

In [ ]:
result = select_as_of(*args)
print(json.dumps(result, ensure_ascii=False, indent=2))

## Checks

Compare every row with the browser example's expected output. After editing inputs, a failed assertion may be expected: explain the difference before changing the check.

In [ ]:
assert result == expected, "Output differs from the reference synthetic example"
assert bundle["identity"] == "synthetic"
print("Passed: output matches the synthetic browser example.")

## Next steps

Before real data, confirm grants, fields, schema_major, windows and provenance using authenticated GET /v1/catalog, then map the actual contract. Candidate IDs below do not guarantee availability or historical completeness. API as_of is not a historical filing-version guarantee. Validate again after substituting real inputs; the synthetic pass does not transfer.

- `cn.dataset.income`
- `cn.dataset.cashflow`
- `cn.dataset.balancesheet`

### References

- [Tushare: financial-statement data](https://tushare.pro/document/2?doc_id=16)
- [Dechow & Dichev: accrual estimation errors](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=277231)

[Back to tutorial](https://tradingdatas.com/recipes/pit-fundamentals-panel/)